<center><h1>Annotations for the Dataset Generator</h1></center>
<center><h4>Going over all the code from everyone and splitting it up into a legible and usable base to generate the artifical markets for the Black-Sholes Training</h4></center>

### Imports

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import t as t_dist
from arch import arch_model
import yfinance as yf
import time
import os
import json
import hashlib
from cir_model import CIRModel
from scipy.stats import truncnorm

### Miscellaneous Functions and Global Variables for Reproducibility

In [2]:
MASTER_SEED = 42  # Change this to generate different datasets

def set_all_seeds(seed):
    """Set all random seeds for reproducibility"""
    np.random.seed(seed)
    print(f"All seeds set to: {seed}")

def compute_dataframe_hash(df, sample_size=10000):
    """
    Compute hash of dataframe to verify reproducibility.
    Uses a sample to avoid memory issues with large datasets.
    """
    if len(df) > sample_size:
        df_sample = df.sample(n=sample_size, random_state=42).sort_index()
    else:
        df_sample = df
    
    return hashlib.md5(pd.util.hash_pandas_object(df_sample).values).hexdigest()

def save_metadata(metadata, filename='dataset_metadata.json'):
    """Save generation metadata for verification"""
    with open(filename, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"\nMetadata saved to {filename}")

#### Setting the seed in order to reproduce the same numbers

In [3]:
# Set seed before any random operations
set_all_seeds(MASTER_SEED)

start_date = "2019-01-01"
end_date = "2025-01-01"

All seeds set to: 42


### Downloading FTSE 100 Market Data and Fitting Garch model
Using a Garch(1,1)-t model, which is defined with returns as
$$r_t=\mu + \varepsilon_t$$
with the residuals
$$\varepsilon_t=\sigma_tz_t$$
and the resurcive variance equation
$$\sigma_t^2 = \omega + \alpha\varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$
which put into infinite form is
$$\sigma_t^2=\frac{\omega}{1-\beta}+\sum_{k=1}^{\infty}\alpha\beta^{k-1}\varepsilon_{t-k}^2$$
Given $\sigma^2_t = $ conditional variance, and
$$z_t\sim \text{i.i.d. } t_v(0,1)$$
and the constraints
$$\omega > 0\\ \alpha \geq 0 \\ \beta \geq 0\\ \alpha + \beta < 1$$
So in total for parameters, we have
$$\theta = (\mu, \omega, \alpha, \beta, v)$$

In [4]:
data = yf.download("^FTSE", start=start_date, end=end_date)

if isinstance(data.columns, pd.MultiIndex):
    close_prices = data['Close']['^FTSE']
else:
    close_prices = data['Close']

S0 = float(close_prices.iloc[-1])

print(f"Downloaded {len(close_prices):,} trading days")
print(f"S_0 (initial stock price): £{S0:,.2f}")

# Fit GARCH(1,1)-t
returns_pct = 100 * np.log(close_prices / close_prices.shift(1)).dropna()

print("\nFitting GARCH(1,1)-t model...")
model = arch_model(returns_pct, vol='Garch', p=1, q=1, dist='t')
result = model.fit(disp='off')

mu = result.params['mu']
omega = result.params['omega']
alpha = result.params['alpha[1]']
beta = result.params['beta[1]']
nu = result.params['nu']

print(f"Parameters: μ={mu:.4f}%, ω={omega:.6f}, α={alpha:.4f}, β={beta:.4f}, ν={nu:.2f}")

[*********************100%***********************]  1 of 1 completed

Downloaded 1,514 trading days
S_0 (initial stock price): £8,173.00

Fitting GARCH(1,1)-t model...
Parameters: μ=0.0484%, ω=0.046704, α=0.1335, β=0.8205, ν=4.60


### Parameterizing a Cox–Ingersoll–Ross Model 
The CIR model is defined by the SDE
$$dr_t = \kappa(\theta - r_t)dt + \sigma\sqrt{r_t}dW_t$$
where $\kappa(\theta - r_t)$ serves as the drift term with a long-run mean of $\theta$ and a mean reversion speed of $\kappa$. $W_t$ is a standard Wiener process.

The volatility term $\sigma\sqrt{r_t}$ creates level dependent volatility, which dampens volatility when $r_t$ is close to 0. If the Feller condition is met, defined by
$$2\kappa \theta \geq \sigma ^2$$
then negative interest rates are impossible.

In the code we parameterize our CIR model on past daily SONIA data provided by the U.S. Federal Reserve, previously downloaded

In [5]:
# Getting a dataframe of historical interest rates
historical_IR_sonia = pd.read_csv("parameterize_data\IUDSOIA.csv")
historical_IR_sonia["observation_date"] = pd.to_datetime(historical_IR_sonia["observation_date"])
historical_ir_wanted = historical_IR_sonia[(historical_IR_sonia["observation_date"] >= start_date) & (historical_IR_sonia["observation_date"] <= end_date)].dropna()

print(historical_ir_wanted.head())
# Using the CIR Model Class in the repo
cir = CIRModel(historical_ir_wanted)
cir.calibrate(method='mle')

     observation_date  interest_rate
5739       2019-01-02         0.7044
5740       2019-01-03         0.7048
5741       2019-01-04         0.7046
5742       2019-01-07         0.7052
5743       2019-01-08         0.7052
Loaded 1515 data points
Date range: 2019-01-02 00:00:00 to 2024-12-31 00:00:00
Mean rate: 200.2999%
MLE optimization failed, falling back to method of moments

Calibrated parameters (Method of Moments):
  κ (kappa): 0.0557 - mean reversion speed
  θ (theta): 200.2999% - long-term mean
  σ (sigma): 0.4673 - volatility
  Feller condition (2κθ > σ²): 0.223076 > 0.218387 = True


{'kappa': 0.05568541895992372,
 'theta': 2.002999471947195,
 'sigma': 0.4673194433258763}

### Saving the model parameters in order to load them for later without using having to reparameterize the models

In [6]:
params = {
    "End Price" : S0,
    "Mean IR" : cir.historical_data['interest_rate'].mean(),
    "IR std" : np.std(cir.historical_data['interest_rate']),
    "GARCH" : {
        "mu": mu,
        "omega": omega,
        "alpha": alpha,
        "beta": beta,
        "nu": nu
    },
    "CIR": {
        "kappa": cir.kappa,
        "theta": cir.theta,
        "sigma": cir.sigma
    }
}

with open("parameterize_data/model_params.json", "w") as f:
    json.dump(params, f, indent=4)
    
del mu, omega, alpha, beta, nu, cir, data

print("model_params.json written successfully.")

model_params.json written successfully.


### Reloading the model simulation parameters, so code can be executed beyond this line

In [7]:
with open("parameterize_data/model_params.json", "r") as f:
    params = json.load(f)

# Historical Parameters
S0 = params["End Price"]
ir_mean = params["Mean IR"]
ir_std = params["IR std"]


# GARCH parameters
mu    = params["GARCH"]["mu"]
omega = params["GARCH"]["omega"]
alpha = params["GARCH"]["alpha"]
beta  = params["GARCH"]["beta"]
nu     = params["GARCH"]["nu"]

# CIR parameters
kappa = params["CIR"]["kappa"]
theta = params["CIR"]["theta"]
sigma = params["CIR"]["sigma"]

print("Option Price")
print(f"  S0 = {S0}")

print("\nInterest Rates")
print(f"  Mean Interest Rate = {ir_mean}")
print(f"  Interest Rate Standard Deviation = {ir_std}")

print("\nGARCH Parameters:")
print(f"  mu    (μ) = {mu}")
print(f"  omega (ω) = {omega}")
print(f"  alpha (α) = {alpha}")
print(f"  beta  (β) = {beta}")
print(f"  nu    (ν) = {nu}")

print("\nCIR Parameters:")
print(f"  kappa (κ) = {kappa}")
print(f"  theta (θ) = {theta}")
print(f"  sigma (σ) = {sigma}")

Option Price
  S0 = 8173.0

Interest Rates
  Mean Interest Rate = 2.002999471947195
  Interest Rate Standard Deviation = 2.1061137312665377

GARCH Parameters:
  mu    (μ) = 0.04842674175412159
  omega (ω) = 0.04670376471231804
  alpha (α) = 0.1334992945018068
  beta  (β) = 0.8204776734410293
  nu    (ν) = 4.6037768961503795

CIR Parameters:
  kappa (κ) = 0.05568541895992372
  theta (θ) = 2.002999471947195
  sigma (σ) = 0.4673194433258763


### Setting up the Simulation Parameters and Saved Metadata for reproducibility 

In [9]:
n_simulations = 100000
T_maturity = 5  # All simulations start at 5 years
K_percentages = np.array([0.60, 0.70, 0.80, 0.90, 1.00, 1.10, 1.20, 1.30, 1.40, 1.50])
days_per_year = 252
n_days = T_maturity * days_per_year  # Total days for each simulation
mu_adjusted = 0.04
min_starting_ir = 0.01

print(f"Total simulations: {n_simulations:,}")
print(f"Maturity (T): {T_maturity} years for ALL simulations")
print(f"Days per simulation: {n_days}")
print(f"K choices: {K_percentages * 100}%")

# Store metadata for reproducibility
metadata = {
    'master_seed': MASTER_SEED,
    'n_simulations': int(n_simulations),
    'T_maturity': int(T_maturity),
    'K_percentages': K_percentages.tolist(),
    'S0': float(S0),
    'days_per_year': int(days_per_year),
    'n_days': int(n_days),
    'mu_adjusted': float(mu_adjusted),
    'data_download': {
        'ticker': '^FTSE',
        'start_date': start_date,
        'end_date': end_date,
        'n_days': int(len(close_prices))
    },
    'GARCH_params': {
        "mu": mu,
        "omega": omega,
        "alpha": alpha,
        "beta": beta,
        "nu": nu
    },
    "CIR_params" : {
        "kappa": kappa,
        "theta": theta,
        "sigma": sigma
    },
    "IR_params" : {
        "IR Mean" : ir_mean,
        "IR Standard Deviation" : ir_std,
        "Minimum IR Start" : min_starting_ir
    },
    'python_version': os.sys.version,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'generation_timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

Total simulations: 100,000
Maturity (T): 5 years for ALL simulations
Days per simulation: 1260
K choices: [ 60.  70.  80.  90. 100. 110. 120. 130. 140. 150.]%


### Creating Strike Prices for the Whole Dataset

In [10]:
# CRITICAL: Set seed before random assignments
np.random.seed(MASTER_SEED)

# Each simulation gets ONE randomly assigned strike price
K_pct_assignments = np.random.choice(K_percentages, size=n_simulations)
K_assignments = S0 * K_pct_assignments

print(f"K distribution:")
for k_pct in K_percentages:
    count = np.sum(K_pct_assignments == k_pct)
    print(f"  K={k_pct*100:.0f}%: {count:,} ({count/n_simulations*100:.1f}%)")

K distribution:
  K=60%: 9,974 (10.0%)
  K=70%: 9,911 (9.9%)
  K=80%: 9,886 (9.9%)
  K=90%: 9,984 (10.0%)
  K=100%: 9,975 (10.0%)
  K=110%: 10,054 (10.1%)
  K=120%: 10,104 (10.1%)
  K=130%: 9,925 (9.9%)
  K=140%: 10,031 (10.0%)
  K=150%: 10,156 (10.2%)


### Simulating GARCH Price paths for the whole dataset
What this code is doing is creating two seperate multidimensional arrays through numpy to track volatility and price over time. We have two matrices $S_{ij}$ and $\sigma_{ij}$. Where $i$ is the number of scenarios we are running and $j$ is the number of days per scenario. Our initial volatility is given by
$$ \sigma^2_0 = \frac{\omega}{1-\alpha - \beta} $$
which is the expectation of variance. We set our initial positions for the simulations at 
$$\sigma_{i0} = \sqrt{\sigma^2}, \text{ as defined above}$$
$$S_{i0}=S0, \text{ the end of the stock data price time series we parameterized the GARCH on}$$
A Monte Carlo simulation is now preformed using the GARCH, where
$$z_{i,t} \sim t_v$$
Which we normalize so
$$\text{Var}(z_{i,t})=1$$
Which we do by letting
$$z_{i,t}=\frac{\tilde{z}_{i,t}}{\sqrt{\frac{v}{v-2}}}$$
Since the normal t distribution variance is given by
$$\text{Var}(t_v) = \frac{v}{v-2}$$
The rest is just calculating the current variables using the equations described earlier for GARCH. Two things to note however are that
1. A multiplicative log-return process is used
$$S_{i,t} = S_{i,t-1} \cdot \text{exp}\left(\frac{r_{i,t}}{100}\right)$$
2. Volatility is capped at 25%

In [11]:
def simulate_garch_vectorized(S0, n_days, n_scenarios, mu, omega, alpha, beta, nu, seed=None):
    """
    Fully vectorized GARCH(1,1)-t simulation with seed control.
    
    Args:
        seed: If provided, sets numpy seed before simulation
    """
    if seed is not None:
        np.random.seed(seed)
    
    var0 = omega / (1 - alpha - beta) # Initial volatility condition of the GARCH model
    
    # Two multidemensional
    S_paths = np.zeros((n_scenarios, n_days)) # The shape is rows, then columns, so the number of rows is the number of simulations and the number of columns is the number of days
    sigma_paths = np.zeros((n_scenarios, n_days))
    
    # Setting the first column to all the initial starting conditions needed
    S_paths[:, 0] = S0 # Filling the First 
    sigma_paths[:, 0] = np.sqrt(var0)
    
    # Variables to measure current price and volatility
    var_current = np.full(n_scenarios, var0)
    S_current = np.full(n_scenarios, S0)
    
    for day in range(1, n_days): # Running through volatility over time
        z = t_dist.rvs(df=nu, size=n_scenarios) / np.sqrt(nu / (nu - 2)) # Standardizing shock so the variance is one
        
        # Standard Calculations
        sigma_current = np.sqrt(var_current)
        eps = sigma_current * z 
        return_pct = mu + eps
        
        S_current = S_current * np.exp(return_pct / 100) # Setting a new current price by using the old prices variable times a log return
        
        # Updating the matrix
        S_paths[:, day] = S_current
        sigma_paths[:, day] = sigma_current
        
        var_current = omega + alpha * (eps ** 2) + beta * var_current # Next volatility calculation
        var_current = np.minimum(var_current, 25.0) # Capping volatility to prevent runaway 
    
    return S_paths, sigma_paths


### Simulating interest rates using CIR
Some notes about the methods:
1. The starting interest rate is sampled from a truncated normal distribution between (0.01, 25). 
$$X_{i,0} = \text{TruncNormal}(\mu_0, \sigma^2_0; 0.01, 25.0)$$
Which means it has a PDF of
$$f(x) = \frac{\phi\left(\frac{x-\mu_0}{\sigma_0}\right)}{\sigma\left[\Phi(b)-\Phi(a)\right]} \qquad \text{for $x \in [0.01, 10]$}$$
where in this case
$$a = \Phi\left(\frac{0.01-\mu_0}{\sigma_0}\right), \qquad b=\Phi\left(\frac{25-\mu_0}{\sigma_0}\right)$$
2. Using the Euler-Maruyama Method
$$dr_t = \kappa (\theta - r_t)dt+\sigma\sqrt{r_t}dW_t$$
we turn into
$$r_{t+1}=r_t+\kappa(\theta-r_t)\Delta t + \sigma\sqrt{r_t}\sqrt{\Delta t}Z_t$$
where
$$Z_t \sim N(0,1)$$

In [12]:
def simulate_CIR_paths(n_scenarios,n_steps,kappa,theta,sigma,mu0,sigma0_init,dt=1/252,seed=None):
    if seed is not None:
        np.random.seed(seed)
    
    # ---- Truncated normal initial condition ----
    lower, upper = 0.01, 25.0
    
    a = (lower - mu0) / sigma0_init
    b = (upper - mu0) / sigma0_init
    
    X0_samples = truncnorm.rvs(
        a, b,
        loc=mu0,
        scale=sigma0_init,
        size=n_scenarios
    )
    
    # ---- Storage ----
    X_paths = np.zeros((n_scenarios, n_steps))
    X_paths[:, 0] = X0_samples
    X_current = X0_samples.copy()
    
    # ---- Time stepping ----
    for t in range(1, n_steps):
        
        z = np.random.normal(size=n_scenarios)
        
        drift = kappa * (theta - X_current) * dt
        diffusion = sigma * np.sqrt(np.maximum(X_current, 0.0)) * np.sqrt(dt) * z
        
        X_next = X_current + drift + diffusion
        
        # Enforce positivity
        X_next = np.maximum(X_next, 0.0)
        
        X_paths[:, t] = X_next
        X_current = X_next
    
    return X_paths

### Running the Simulations

In [13]:
# =============================================================================
# Sorting out saving directory
# =============================================================================

output_dir = 'black_scholes_simulation_data'
os.makedirs(output_dir, exist_ok=True)

total_start = time.time()

print(f"\nGenerating {n_simulations:,} simulations × {n_days} days")

# =============================================================================
# Running GARCH
# =============================================================================

print(f"Running GARCH simulation...")
sim_start = time.time()

# CRITICAL: Use deterministic seed
simulation_seed = MASTER_SEED + 5000

S_paths, sigma_paths = simulate_garch_vectorized(
    S0=S0,
    n_days=n_days,
    n_scenarios=n_simulations,
    mu=mu_adjusted,
    omega=omega,
    alpha=alpha,
    beta=beta,
    nu=nu,
    seed=simulation_seed
)

print(f"GARCH simulation completed in {time.time() - sim_start:.1f}s")

# =============================================================================
# Running CIR Model
# =============================================================================

print(f"Running CIR simulation...")
sim_start = time.time()

# Deterministic seed for CIR
cir_seed = MASTER_SEED + 7000

interest_paths = simulate_CIR_paths(
    n_scenarios=n_simulations,
    n_steps=n_days,
    kappa=kappa,
    theta=theta,
    sigma=sigma,
    mu0=ir_mean,
    sigma0_init=ir_std,
    dt=1/252,
    seed=cir_seed
)

print(f"CIR simulation completed in {time.time() - sim_start:.1f}s")

# =============================================================================
# Building a vectorized Dataset 
# =============================================================================

print(f"\nBuilding dataset...")
build_start = time.time()

n_rows = n_simulations * n_days

# Create arrays for each column
simulation_col = np.repeat(np.arange(n_simulations), n_days)
day_col = np.tile(np.arange(n_days), n_simulations)
S_col = S_paths.flatten()
K_col = np.repeat(K_assignments, n_days)
interest_col = interest_paths.flatten()

# Create T column: starts at T_maturity and decreases each day
T_sequence = T_maturity - np.arange(n_days) / days_per_year
T_col = np.tile(T_sequence, n_simulations)

# Convert volatility to annualized percentage
sigma_col = sigma_paths.flatten() * np.sqrt(days_per_year) / 100

print(f"Arrays built in {time.time() - build_start:.1f}s")

# =============================================================================
# Creating the DataFrame
# =============================================================================

print(f"Creating DataFrame...")
df_start = time.time()

df = pd.DataFrame({
    'simulation': simulation_col,
    'day': day_col,
    'S': S_col,
    'K': K_col,
    'T': T_col,
    'sigma': sigma_col,
    'r': interest_col
})

# Sort by simulation, then descending T (day ascending)
df = df.sort_values(
    by=['simulation', 'day'],
    ascending=[True, True]
).reset_index(drop=True)

print(f"DataFrame created in {time.time() - df_start:.1f}s")

total_time = time.time() - total_start
print(f"\n" + "=" * 70)
print(f"SIMULATION COMPLETE! Total time: {total_time / 60:.1f} minutes")
print("=" * 70)

# =============================================================================
# Saving
# =============================================================================

print("\n" + "=" * 70)
print("STEP 5: FINAL DATASET")
print("=" * 70)

print("\nStatistics:")
print(df.describe())

# Compute final hash
print("\nComputing dataset hash...")
final_hash = compute_dataframe_hash(df)
print(f"Final dataset hash: {final_hash}")

# Save final file as CSV
print("\nSaving final dataset...")
save_start = time.time()
df.to_csv('black_scholes_simulation_data_T5.csv', index=False)
save_time = time.time() - save_start

file_size_gb = os.path.getsize('black_scholes_simulation_data_T5.csv') / 1e9
print(f"Saved to black_scholes_simulation_data_T5.csv in {save_time:.1f}s")
print(f"File size: {file_size_gb:.2f} GB")

# =============================================================================
# STEP 7: SAVE METADATA AND VERIFICATION INFO
# =============================================================================

print("\n" + "=" * 70)
print("STEP 6: SAVING METADATA FOR REPRODUCIBILITY")
print("=" * 70)

# Add hashes and final statistics to metadata
metadata['final_hash'] = final_hash
metadata['total_rows'] = int(len(df))
metadata['file_size_gb'] = float(file_size_gb)
metadata['total_generation_time_minutes'] = float(total_time / 60)
metadata['simulation_seed'] = int(simulation_seed)
metadata['cir_seed'] = int(cir_seed)

# Save metadata
save_metadata(metadata, f'{output_dir}/dataset_metadata.json')

# Also save a verification file with just the hash
verification = {
    'master_seed': MASTER_SEED,
    'simulation_seed': simulation_seed,
    'final_hash': final_hash,
    'total_rows': int(len(df)),
    'n_simulations': int(n_simulations),
    'T_maturity': int(T_maturity),
    'n_days': int(n_days),
    'cir_seed' : int(cir_seed),
    'instructions': f'Run the generation script with MASTER_SEED={MASTER_SEED} to reproduce this exact dataset'
}

with open('REPRODUCIBILITY_INFO.json', 'w') as f:
    json.dump(verification, f, indent=2)

print(f"\nVerification info saved to REPRODUCIBILITY_INFO.json")

print("\n" + "=" * 70)
print("DONE!")
print("=" * 70)
print("\nTo reproduce this dataset on another computer:")
print(f"1. Use MASTER_SEED = {MASTER_SEED}")
print(f"2. Expected final hash: {final_hash}")
print(f"3. Expected total rows: {len(df):,}")
print(f"4. Expected file size: {file_size_gb:.2f} GB")
print("=" * 70)

# Clean up memory
del S_paths, sigma_paths, interest_paths


Generating 100,000 simulations × 1260 days
Running GARCH simulation...
GARCH simulation completed in 7.4s
Running CIR simulation...
CIR simulation completed in 2.6s

Building dataset...
Arrays built in 1.8s
Creating DataFrame...
DataFrame created in 24.3s

SIMULATION COMPLETE! Total time: 0.6 minutes

STEP 5: FINAL DATASET

Statistics:
         simulation           day             S             K             T  \
count  1.260000e+08  1.260000e+08  1.260000e+08  1.260000e+08  1.260000e+08   
mean   4.999950e+04  6.295000e+02  1.097747e+04  8.594367e+03  2.501984e+00   
std    2.886751e+04  3.637306e+02  3.561307e+03  2.348827e+03  1.443375e+00   
min    0.000000e+00  0.000000e+00  1.963106e+03  4.903800e+03  3.968254e-03   
25%    2.499975e+04  3.147500e+02  8.546233e+03  6.538400e+03  1.252976e+00   
50%    4.999950e+04  6.295000e+02  1.005024e+04  8.990300e+03  2.501984e+00   
75%    7.499925e+04  9.442500e+02  1.246911e+04  1.062490e+04  3.750992e+00   
max    9.999900e+04  1.259000

In [14]:
myDataset = pd.read_csv("black_scholes_simulation_data_T5.csv")

In [18]:
myDataset.head(1000)

,simulation,day,S,K,T,sigma,r
0,0,0,8173.000000,9807.6,5.000000,0.159915,3.145543
1,0,1,8193.466161,9807.6,4.996032,0.159915,3.075683
2,0,2,7884.383581,9807.6,4.992063,0.149356,3.145157
3,0,3,7917.917051,9807.6,4.988095,0.265073,3.107774
4,0,4,7936.117904,9807.6,4.984127,0.243565,3.113102
...,...,...,...,...,...,...,...
995,0,995,5350.032835,9807.6,1.051587,0.136940,5.472184
996,0,996,5363.623981,9807.6,1.047619,0.139309,5.420211
997,0,997,5356.610403,9807.6,1.043651,0.131353,5.429688
998,0,998,5352.958419,9807.6,1.039683,0.124223,5.492090
